In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().resolve()
raw_path = None
for candidate in [project_root, project_root.parent]:
    path = candidate / "data" / "raw" / "Bengaluru_dataset.csv"
    if path.exists():
        raw_path = path
        break

if raw_path is None:
    raise FileNotFoundError("Bengaluru_dataset.csv not found under data/raw/.")

print("Using dataset:", raw_path)


Using dataset: /workspaces/groundwater-ai/data/raw/Bengaluru_dataset.csv


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


#Load the dataset

In [3]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd().resolve()
raw_path = None
for candidate in [project_root, project_root.parent]:
    path = candidate / "data" / "raw" / "Bengaluru_dataset.csv"
    if path.exists():
        raw_path = path
        break

if raw_path is None:
    raise FileNotFoundError("Bengaluru_dataset.csv not found under data/raw/.")

df = pd.read_csv(raw_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("Source:", raw_path)


Dataset loaded successfully!
Shape: (100881, 22)
Source: /workspaces/groundwater-ai/data/raw/Bengaluru_dataset.csv


#Inspect the columns and first records

In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

Columns:
['_id', 'SlNo', 'Station', 'Agency', 'State LGD Code', 'State', 'District LGD Code', 'District', 'Tehsil', 'Block', 'Village', 'River', 'Basin', 'Tributary', 'Subtributary', 'SubSubtributary', 'Local River', 'Latitude', 'Longitude', 'RL_MSL', 'Data Acquisition Time', 'Groundwater Level Telemetry 6 Hourly (meter)']

First 5 records:


,_id,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,River,Basin,Tributary,Subtributary,SubSubtributary,Local River,Latitude,Longitude,RL_MSL,Data Acquisition Time,Groundwater Level Telemetry 6 Hourly (meter)
0,4553464,4553464,Rajanukunte,Karnataka GW,29,Karnataka,525,Bangalore Urban,Yelahanka,Yelahanka,Rajanukunte,-,-,-,-,-,-,13.17,77.57,NaN,22-11-2022 12:00,-24.029
1,4553465,4553465,Rajanukunte,Karnataka GW,29,Karnataka,525,Bangalore Urban,Yelahanka,Yelahanka,Rajanukunte,-,-,-,-,-,-,13.17,77.57,NaN,22-11-2022 18:00,-24.054
2,4553466,4553466,Rajanukunte,Karnataka GW,29,Karnataka,525,Bangalore Urban,Yelahanka,Yelahanka,Rajanukunte,-,-,-,-,-,-,13.17,77.57,NaN,23-11-2022 00:00,-24.047
3,4553467,4553467,Rajanukunte,Karnataka GW,29,Karnataka,525,Bangalore Urban,Yelahanka,Yelahanka,Rajanukunte,-,-,-,-,-,-,13.17,77.57,NaN,23-11-2022 06:00,-24.095
4,4553468,4553468,Rajanukunte,Karnataka GW,29,Karnataka,525,Bangalore Urban,Yelahanka,Yelahanka,Rajanukunte,-,-,-,-,-,-,13.17,77.57,NaN,23-11-2022 12:00,-24.120


#Check data types and missing values

In [5]:
print("DATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

DATA TYPES
_id                                               int64
SlNo                                              int64
Station                                             str
Agency                                              str
State LGD Code                                    int64
State                                               str
District LGD Code                                 int64
District                                            str
Tehsil                                              str
Block                                               str
Village                                             str
River                                               str
Basin                                               str
Tributary                                           str
Subtributary                                        str
SubSubtributary                                     str
Local River                                         str
Latitude                             

#Check the timestamp properly

In [6]:
print("Timestamp range before conversion:")
print("First:", df["Data Acquisition Time"].min())
print("Last :", df["Data Acquisition Time"].max())

print("\nSample timestamps:")
print(df["Data Acquisition Time"].head(10).to_string(index=False))

Timestamp range before conversion:
First: 01-01-2022 00:00
Last : 31-12-2023 18:00

Sample timestamps:
22-11-2022 12:00
22-11-2022 18:00
23-11-2022 00:00
23-11-2022 06:00
23-11-2022 12:00
23-11-2022 18:00
24-11-2022 00:00
24-11-2022 06:00
24-11-2022 12:00
24-11-2022 18:00


#Convert timestamp + check time intervals

In [7]:
# Convert timestamp to datetime
df["Data Acquisition Time"] = pd.to_datetime(
    df["Data Acquisition Time"],
    format="%d-%m-%Y %H:%M"
)

print("Timestamp converted successfully.")
print("Data type:", df["Data Acquisition Time"].dtype)

# Sort by station and timestamp
df = df.sort_values(
    ["Station", "Data Acquisition Time"]
).reset_index(drop=True)

# Calculate time difference within each station
df["time_diff"] = (
    df.groupby("Station")["Data Acquisition Time"]
      .diff()
)

print("\nTime interval distribution:")
print(df["time_diff"].value_counts().head(15))

Timestamp converted successfully.
Data type: datetime64[us]



Time interval distribution:
time_diff
0 days 06:00:00     95381
1 days 06:00:00      2053
0 days 00:00:00      1434
2 days 06:00:00       746
3 days 06:00:00       387
4 days 06:00:00       163
0 days 12:00:00       151
5 days 06:00:00       107
7 days 06:00:00       100
8 days 06:00:00        93
6 days 06:00:00        81
10 days 06:00:00       22
16 days 06:00:00       20
0 days 18:00:00        14
1 days 12:00:00        13
Name: count, dtype: int64


#Measure data completeness station-by-station

Before cleaning anything, let's find out which stations have good continuous coverage and which have large gaps.

This tells us whether all 25 stations are equally useful.

In [8]:
# Calculate number of observations and time coverage for each station
station_summary = df.groupby("Station").agg(
    observations=("Data Acquisition Time", "count"),
    first_record=("Data Acquisition Time", "min"),
    last_record=("Data Acquisition Time", "max")
).reset_index()

# Calculate expected 6-hour observations for each station
station_summary["days_covered"] = (
    station_summary["last_record"] - station_summary["first_record"]
).dt.total_seconds() / (24 * 60 * 60)

station_summary["expected_6hr_records"] = (
    station_summary["days_covered"] * 4
).round().astype(int) + 1

station_summary["coverage_percent"] = (
    station_summary["observations"] /
    station_summary["expected_6hr_records"] * 100
)

station_summary = station_summary.sort_values(
    "coverage_percent",
    ascending=False
)

display(station_summary)

,Station,observations,first_record,last_record,days_covered,expected_6hr_records,coverage_percent
12,K Narayanapura,3317,2023-01-25 06:00:00,2025-12-30 18:00:00,1070.50,4283,77.445716
4,Bagalagunte,4684,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,75.257069
20,Sarjapura_1,4657,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.823265
13,Kethohalli,4656,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.807198
7,Chandapura_1,4622,2021-09-29 00:00:00,2025-12-30 18:00:00,1553.75,6216,74.356499
24,Yelahanka_1,4623,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.276992
6,Byadarahalli,4612,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.100257
14,Laggere,4600,2021-09-28 06:00:00,2025-12-30 18:00:00,1554.50,6219,73.966876
0,Adakamaranahalli,4596,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,73.843188
22,Thimmenahalli,4583,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,73.634319


#Inspect the duplicate records

Need to see what the duplicates actually contain.

In [9]:
# Find duplicate Station + Timestamp combinations
duplicates = df[
    df.duplicated(
        subset=["Station", "Data Acquisition Time"],
        keep=False
    )
].sort_values(
    ["Station", "Data Acquisition Time"]
)

print("Total duplicate rows:", len(duplicates))
print(
    "Duplicate Station + Timestamp combinations:",
    duplicates.groupby(
        ["Station", "Data Acquisition Time"]
    ).ngroups
)

display(duplicates.head(20))

Total duplicate rows: 2868
Duplicate Station + Timestamp combinations: 1434


,_id,SlNo,Station,Agency,State LGD Code,State,District LGD Code,District,Tehsil,Block,Village,River,Basin,Tributary,Subtributary,SubSubtributary,Local River,Latitude,Longitude,RL_MSL,Data Acquisition Time,Groundwater Level Telemetry 6 Hourly (meter),time_diff
7603,401548,401548,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 00:00:00,-93.592,NaT
7604,401548,401548,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 00:00:00,-93.592,0 days 00:00:00
7605,401549,401549,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 06:00:00,-93.589,0 days 06:00:00
7606,401549,401549,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 06:00:00,-93.589,0 days 00:00:00
7607,401550,401550,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 12:00:00,-93.567,0 days 06:00:00
7608,401550,401550,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 12:00:00,-93.567,0 days 00:00:00
7609,401551,401551,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 18:00:00,-93.585,0 days 06:00:00
7610,401551,401551,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-05-29 18:00:00,-93.585,0 days 00:00:00
7611,401552,401552,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-08-10 00:00:00,-93.543,72 days 06:00:00
7612,401552,401552,Attibele_1,Karnataka GW,29,Karnataka,525,Bangalore Urban,-,-,-,-,-,-,-,-,-,12.778781,77.772672,883.0,2022-08-10 00:00:00,-93.543,0 days 00:00:00


#Investigate and handle duplicate observations



## 2.5 Duplicate Observation Analysis

Before performing data cleaning, duplicate observations are investigated to determine whether the dataset contains repeated measurements for the same monitoring station and timestamp.

Since the groundwater telemetry data is expected to be recorded at six-hour intervals, multiple records having the same `Station` and `Data Acquisition Time` may represent duplicated observations rather than independent measurements.

The duplicates are inspected before removal so that valid observations are not accidentally discarded.

### Purpose
- Identify repeated Station–Timestamp combinations.
- Determine whether duplicated records contain identical groundwater-level measurements.
- Establish an appropriate duplicate-handling strategy for subsequent analysis.

In [10]:
# Check whether duplicate Station-Timestamp records have identical values

duplicate_groups = (
    duplicates
    .groupby(["Station", "Data Acquisition Time"])
    ["Groundwater Level Telemetry 6 Hourly (meter)"]
    .nunique()
)

print("Duplicate groups with identical groundwater values:",
      (duplicate_groups == 1).sum())

print("Duplicate groups with different groundwater values:",
      (duplicate_groups > 1).sum())

print("\nTotal duplicate groups:", len(duplicate_groups))

Duplicate groups with identical groundwater values: 1434
Duplicate groups with different groundwater values: 0

Total duplicate groups: 1434


### Findings

The duplicate analysis identified **1,434 duplicated Station–Timestamp groups**, corresponding to **2,868 duplicate rows**.

All 1,434 duplicate groups contain identical groundwater-level values. No duplicate group was found with conflicting groundwater measurements.

Therefore, these records are confirmed as exact duplicate observations rather than separate measurements.

### Decision

For subsequent analysis, one copy of each duplicated Station–Timestamp observation will be retained and the exact duplicate rows will be removed.

This prevents duplicated measurements from artificially increasing the importance of particular observations while preserving all unique groundwater-level records.

## 2.6 Removal of Exact Duplicate Observations

Based on the duplicate analysis, the identified duplicate records are exact copies with identical Station, timestamp, and groundwater-level measurements.

Therefore, duplicate rows are removed using the combination of:

- `Station`
- `Data Acquisition Time`
- `Groundwater Level Telemetry 6 Hourly (meter)`

Only one observation is retained for each unique Station–Timestamp combination.

The original `df` is preserved, and a new dataframe named `df_clean` is created for subsequent analysis.

In [11]:
# Create a cleaned copy without exact duplicate observations

df_clean = df.drop_duplicates(
    subset=[
        "Station",
        "Data Acquisition Time",
        "Groundwater Level Telemetry 6 Hourly (meter)"
    ],
    keep="first"
).copy()

print("Original rows :", len(df))
print("Cleaned rows  :", len(df_clean))
print("Rows removed  :", len(df) - len(df_clean))

print(
    "\nRemaining duplicate Station-Timestamp groups:",
    df_clean.duplicated(
        subset=["Station", "Data Acquisition Time"]
    ).sum()
)

Original rows : 100881
Cleaned rows  : 99447
Rows removed  : 1434

Remaining duplicate Station-Timestamp groups: 0


### Findings

After removing the confirmed exact duplicates:

- Original observations: **100,881**
- Duplicate observations removed: **1,434**
- Remaining observations: **99,447**
- Remaining duplicate Station–Timestamp combinations: **0**

The cleaned dataset therefore contains one unique observation for each Station–Timestamp combination.

The original dataset is preserved in `df`, while `df_clean` will be used for subsequent analysis.

## 2.7 Station-wise Data Coverage Analysis

After removing exact duplicate observations, station-wise data coverage is recalculated.

Because the telemetry system is designed to provide measurements at six-hour intervals, the number of available observations is compared with the expected number of six-hour observations between each station's first and last recorded timestamp.

This analysis helps determine the reliability and completeness of each monitoring station's time series before selecting data for predictive modeling.

In [12]:
# Recalculate station-wise coverage using the cleaned dataset

station_summary_clean = df_clean.groupby("Station").agg(
    observations=("Data Acquisition Time", "count"),
    first_record=("Data Acquisition Time", "min"),
    last_record=("Data Acquisition Time", "max")
).reset_index()

# Duration covered by each station
station_summary_clean["days_covered"] = (
    station_summary_clean["last_record"]
    - station_summary_clean["first_record"]
).dt.total_seconds() / (24 * 60 * 60)

# Expected observations assuming six-hour telemetry
station_summary_clean["expected_6hr_records"] = (
    station_summary_clean["days_covered"] * 4
).round().astype(int) + 1

# Coverage percentage
station_summary_clean["coverage_percent"] = (
    station_summary_clean["observations"]
    / station_summary_clean["expected_6hr_records"]
    * 100
)

station_summary_clean = station_summary_clean.sort_values(
    "coverage_percent",
    ascending=False
).reset_index(drop=True)

display(station_summary_clean)

,Station,observations,first_record,last_record,days_covered,expected_6hr_records,coverage_percent
0,K Narayanapura,3317,2023-01-25 06:00:00,2025-12-30 18:00:00,1070.50,4283,77.445716
1,Bagalagunte,4684,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,75.257069
2,Sarjapura_1,4657,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.823265
3,Kethohalli,4656,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.807198
4,Chandapura_1,4622,2021-09-29 00:00:00,2025-12-30 18:00:00,1553.75,6216,74.356499
5,Yelahanka_1,4623,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.276992
6,Byadarahalli,4612,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,74.100257
7,Laggere,4600,2021-09-28 06:00:00,2025-12-30 18:00:00,1554.50,6219,73.966876
8,Adakamaranahalli,4596,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,73.843188
9,Thimmenahalli,4583,2021-09-27 00:00:00,2025-12-30 18:00:00,1555.75,6224,73.634319


## Conclusion

In this notebook, the Bengaluru Urban groundwater dataset was investigated as the
foundation for our Bengaluru-wide groundwater prediction system. The dataset
contains 22 attributes covering station, geographic, temporal and groundwater-level
information.

We checked the dataset structure, data types, missing values, timestamps, sampling
intervals, station-wise coverage and duplicates. The timestamps were successfully
converted to datetime. Most observations follow the expected 6-hour interval, but
irregular gaps exist.

A total of **2,868 duplicate rows** were found, forming **1,434 duplicate
Station + Timestamp groups**. All duplicate groups had identical groundwater
values, so no conflicting duplicate measurements were found. `RL_MSL` was also
found to contain missing values. Station-wise coverage varied considerably
(approximately **47.85%–77.45%**).

**Decisions:** Bengaluru will remain the study area, the model will use data from
multiple Bengaluru stations, the raw dataset will be preserved, and duplicate
removal, missing-value handling and time-gap treatment will be performed separately
in the preprocessing stage rather than modifying the raw data here.

**Outcome:** The raw dataset and its major quality issues are now understood, and
the project is ready to proceed to **Data Cleaning & Preprocessing**.